# CS F407 — SegNet Training on Kaggle
This notebook trains the drone landing zone segmentation model on Kaggle's free T4 GPU.

**Before running:**
1. Add your src files dataset (the one with all .py files) via Add Data on the right
2. Add your training_set dataset (with images/ and gt/ folders) via Add Data on the right
3. Set Accelerator to GPU T4 x2 in Session options
4. Run All

In [ ]:
# ── Step 1: Explore what Kaggle gave us ──────────────────────────────────────
import os

print('=== /kaggle/input/ structure ===')
for dataset in os.listdir('/kaggle/input'):
    print(f'\n[{dataset}]')
    for root, dirs, files in os.walk(f'/kaggle/input/{dataset}'):
        depth = root.replace(f'/kaggle/input/{dataset}', '').count(os.sep)
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
        if depth < 3:  # only show top 3 levels
            for f in files[:5]:
                print(f'{indent}  {f}')
            if len(files) > 5:
                print(f'{indent}  ... ({len(files)} files total)')

In [ ]:
# ── Step 2: Copy src .py files to /kaggle/working/ ───────────────────────────
import shutil
import glob

WORKING = '/kaggle/working'

# Find all .py files across all input datasets and copy to working dir
py_files_found = glob.glob('/kaggle/input/**/*.py', recursive=True)
# Also grab mission_config.json
json_files_found = glob.glob('/kaggle/input/**/*.json', recursive=True)

copied = []
for f in py_files_found + json_files_found:
    dest = os.path.join(WORKING, os.path.basename(f))
    shutil.copy2(f, dest)
    copied.append(os.path.basename(f))

print('Copied to /kaggle/working/:')
for c in sorted(copied):
    print(f'  {c}')

In [ ]:
# ── Step 3: Find and symlink the training data ───────────────────────────────
# We symlink instead of copying — the images are large (400 x 6000x4000px)
# and copying would waste time and disk space.

# Find images/ directory in input
images_dirs = glob.glob('/kaggle/input/**/images', recursive=True)
label_dirs  = glob.glob('/kaggle/input/**/label_images', recursive=True)

print('Found images dirs:', images_dirs)
print('Found label dirs: ', label_dirs)

if images_dirs:
    # The training_set folder is 2 levels up from images/
    training_set_src = os.path.dirname(images_dirs[0])  # src/training_set
    training_set_dst = os.path.join(WORKING, 'training_set')

    if not os.path.exists(training_set_dst):
        os.symlink(training_set_src, training_set_dst)
        print(f'\nSymlinked: {training_set_src} → {training_set_dst}')
    else:
        print(f'\ntraining_set already exists at {training_set_dst}')

    # Verify
    n_images = len(glob.glob(os.path.join(training_set_dst, 'images', '*.jpg')))
    n_masks  = len(glob.glob(os.path.join(training_set_dst, 'gt', 'semantic', 'label_images', '*.png')))
    print(f'Images found: {n_images}')
    print(f'Masks found:  {n_masks}')
else:
    print('ERROR: Could not find images/ directory — check your dataset is added correctly')

In [ ]:
# ── Step 4: Verify GPU and environment ───────────────────────────────────────
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Verify all src files are present
required = ['config.py', 'model.py', 'dataset.py', 'train.py',
            'geometry.py', 'knowledge_graph.py', 'semantic_brain.py',
            'utils.py', 'main.py', 'mission_config.json']
print('\nSrc files check:')
all_ok = True
for f in required:
    path = os.path.join(WORKING, f)
    ok   = os.path.isfile(path)
    print(f'  {"✓" if ok else "✗"} {f}')
    if not ok:
        all_ok = False

print(f'\n{"All files present — ready to train!" if all_ok else "MISSING FILES — check dataset upload"}')

In [ ]:
# ── Step 5: Train ─────────────────────────────────────────────────────────────
# Change into working dir so relative paths (best_model.pth, mission_config.json)
# resolve correctly — config.py uses __file__ to anchor all paths.
os.chdir(WORKING)
print(f'Working directory: {os.getcwd()}')
print('Starting training...\n')

# Run train.py directly in this process so output streams live to the notebook.
# We use exec() so all the module imports share the same Python process.
import sys
sys.argv = ['train.py']   # clear any notebook argv that might confuse argparse

exec(open('train.py').read())

In [ ]:
# ── Step 6: Verify output and show final metrics ──────────────────────────────
ckpt_path = os.path.join(WORKING, 'best_model.pth')
if os.path.isfile(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / 1e6
    print(f'✓ best_model.pth saved ({size_mb:.1f} MB)')
    
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'  Best epoch     : {ckpt["epoch"]}')
    print(f'  Best val loss  : {ckpt["best_val_loss"]:.4f}')
    print(f'  Val accuracy   : {ckpt["val_metrics"]["pixel_acc"]*100:.2f}%')
    print(f'  Val mIoU       : {ckpt["val_metrics"]["mean_iou"]*100:.2f}%')
    print(f'\nDownload best_model.pth from the Output panel on the right.')
else:
    print('ERROR: best_model.pth not found — training may have failed')